In [1]:
import pandas as pd

path = 'dataset/train.csv'
df = pd.read_csv(path)
df_x = df[['emotion']]
df_x.to_csv('dataset/emotion.csv', index=False, header=False)

In [2]:
import cv2
import numpy as np
import os

path = 'face_images'
os.makedirs(path, exist_ok=True)
data = np.loadtxt('dataset/pixels.csv')

for i in range(data.shape[0]):
    face_array = data[i, :].reshape((48, 48)) # reshape
    cv2.imwrite(path + r'/' + '{}.jpg'.format(i), face_array) # 写图片

In [14]:
import shutil
from tqdm import tqdm

picture_root = 'face_images'
train_path = 'face_images/train'
val_path = 'face_images/val'
os.makedirs(train_path, exist_ok=True)
os.makedirs(val_path, exist_ok=True)

for idx in tqdm(os.listdir(picture_root), desc="Copying images"):
    picture_path = os.path.join(picture_root, idx)
    if not os.path.isfile(os.path.join(picture_root, idx)):
        continue
    num = int(os.path.splitext(idx)[0])
    src_path = os.path.join(picture_root, idx)
    if num < 24000:
        dst_path = os.path.join(train_path, idx)
    else:
        dst_path = os.path.join(val_path, idx)
    shutil.copy2(src_path, dst_path)

Copying images: 100%|███████████████████████████████████████████████████████████| 28711/28711 [00:35<00:00, 809.74it/s]


In [22]:
import glob
import mediapipe as mp
from mediapipe.python.solutions import face_mesh_connections as fmc

emotion_path = r"dataset/emotion.csv"

os.makedirs(train_path, exist_ok=True)
os.makedirs(val_path, exist_ok=True)
df_emotion = pd.read_csv(emotion_path, header=None)

region_connections = {
    "lips": fmc.FACEMESH_LIPS,
    "left_eye": fmc.FACEMESH_LEFT_EYE,
    "right_eye": fmc.FACEMESH_RIGHT_EYE,
    "left_eyebrow": fmc.FACEMESH_LEFT_EYEBROW,
    "right_eyebrow": fmc.FACEMESH_RIGHT_EYEBROW,
    "nose": fmc.FACEMESH_NOSE,
    "face_oval": fmc.FACEMESH_FACE_OVAL,
}
use_regions = ["lips","left_eye","right_eye","left_eyebrow","right_eyebrow","nose","face_oval"]

def connections_to_indices(connections):
    idx=set()
    for a,b in connections:
        idx.add(a); idx.add(b)
    return idx

selected_idx = sorted(set().union(*(connections_to_indices(region_connections[r]) for r in use_regions)))
M = len(selected_idx)
print("Selected landmarks:", M)

mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(
    static_image_mode=True,
    max_num_faces=1,
    min_detection_confidence=0.5
)

def extract_selected_landmarks(gray48, upscale=4):
    h, w = gray48.shape[:2]
    img = cv2.resize(gray48, (w*upscale, h*upscale), interpolation=cv2.INTER_CUBIC)
    H, W = img.shape[:2]

    rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    res = face_mesh.process(rgb)
    if not res.multi_face_landmarks:
        return None

    lm = res.multi_face_landmarks[0].landmark
    pts = np.zeros((M, 2), dtype=np.float32)

    for k, idx in enumerate(selected_idx):
        x = lm[idx].x * W / upscale
        y = lm[idx].y * H / upscale
        pts[k] = [x, y]

    return pts

def process_folder(folder, out_prefix, upscale=4):
    files = sorted(glob.glob(os.path.join(folder, "*.jpg")))
    all_landmarks = []
    all_labels = []
    skipped = 0

    for fp in tqdm(files, desc="Processing"):
        filename = os.path.basename(fp)
        index = int(os.path.splitext(filename)[0])

        label = int(df_emotion.iat[index, 0])

        img = cv2.imread(fp, cv2.IMREAD_GRAYSCALE)
        if img is None:
            skipped += 1
            continue

        if img.shape != (48, 48):
            img = cv2.resize(img, (48, 48))

        pts = extract_selected_landmarks(img, upscale=upscale)
        if pts is None:
            skipped += 1
            continue

        all_landmarks.append(pts)
        all_labels.append(label)

    all_landmarks = np.stack(all_landmarks)
    all_labels = np.array(all_labels)

    np.save(out_prefix+"_landmarks.npy", all_landmarks)
    np.save(out_prefix+"_labels.npy", all_labels)

    print(out_prefix, all_landmarks.shape, all_labels.shape, "skipped:", skipped)
    
def has_face(gray48, upscale=4):
    h, w = gray48.shape[:2]
    if upscale and upscale != 1:
        img = cv2.resize(gray48, (w*upscale, h*upscale), interpolation=cv2.INTER_CUBIC)
    else:
        img = gray48
    rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    res = face_mesh.process(rgb)
    return bool(res.multi_face_landmarks)

def make_ids(folder, out_path, upscale=4):
    files = sorted(glob.glob(os.path.join(folder, "*.jpg")))
    kept_ids = []
    for fp in tqdm(files, desc='Processing'):
        img = cv2.imread(fp, cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
        if img.shape != (48, 48):
            img = cv2.resize(img, (48, 48), interpolation=cv2.INTER_AREA)

        if has_face(img, upscale=upscale):
            sid = os.path.splitext(os.path.basename(fp))[0]  # "24000"
            kept_ids.append(sid)

    kept_ids = np.array(kept_ids, dtype=str)
    np.save(out_path, kept_ids)
    print(out_path, kept_ids.shape)

make_ids(train_path, "train_ids.npy", upscale=4)
make_ids(val_path,   "val_ids.npy",   upscale=4)

process_folder(train_path, "train")
process_folder(val_path, "val")

Selected landmarks: 152


Processing: 100%|███████████████████████████████████████████████████████████████| 24000/24000 [01:48<00:00, 220.54it/s]


train (22015, 152, 2) (22015,) skipped: 1985


Processing: 100%|█████████████████████████████████████████████████████████████████| 4709/4709 [00:21<00:00, 215.27it/s]

val (4317, 152, 2) (4317,) skipped: 392


In [1]:
import torch
from torch.utils import data
import torch.nn as nn
import numpy as np
import pandas as pd

class FaceDataset(data.Dataset):
    def __init__(self, img_dir, landmarks_path, labels_path, ids_path):
        self.root = img_dir

        self.landmarks = np.load(landmarks_path).astype(np.float32)
        self.labels = np.load(labels_path).astype(np.int64)
        self.ids = np.load(ids_path).astype(str)

        lm = self.landmarks
        lm = lm - lm.mean(axis=1, keepdims=True)
        norm = np.linalg.norm(lm, axis=(1, 2), keepdims=True) + 1e-6
        self.landmarks = lm / norm

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, i):
        name = str(self.ids[i])
        img_path = os.path.join(self.root, f"{name}.jpg")

        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        
        if img.shape != (48, 48):
            img = cv2.resize(img, (48, 48), interpolation=cv2.INTER_AREA)

        img = cv2.equalizeHist(img)
        img = (img.reshape(1, 48, 48) / 255.0).astype(np.float32)
        img_t = torch.from_numpy(img)

        lm_t = torch.from_numpy(self.landmarks[i]).float()
        y = torch.tensor(int(self.labels[i]), dtype=torch.long)
        return img_t, lm_t, y

In [2]:
class CNN_LSTM_Fusion(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),               # 24x24
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),               # 12x12
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),  # 128x1x1
            nn.Dropout(0.3) ##
        )
        self.cnn_fc = nn.Linear(128, 128)

        self.lstm = nn.LSTM(
            input_size=2,
            hidden_size=128,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )
        
        self.lm_fc = nn.Sequential(
            nn.Linear(lstm_hidden * 2, 128),
            nn.ReLU(),
            nn.Dropout(0.3)      
        )

        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, img, lm_seq):
        x_img = self.cnn(img).flatten(1)     # (B,128)
        x_img = self.cnn_fc(x_img)           # (B,128)

        out, _ = self.lstm(lm_seq)           # (B,M,2H)
        x_lm = out[:, -1, :]                 # (B,2H)
        x_lm = self.lm_fc(x_lm)              # (B,128)

        x = torch.cat([x_img, x_lm], dim=1)  # (B,256)
        return self.classifier(x)

In [3]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    ce = nn.CrossEntropyLoss()
    for img, lm, y in loader:
        img, lm, y = img.to(device), lm.to(device), y.to(device)
        logits = model(img, lm)
        loss = ce(logits, y)
        loss_sum += loss.item() * y.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    return loss_sum / max(total, 1), correct / max(total, 1)

def train_one_epoch(model, loader, optim):
    model.train()
    ce = nn.CrossEntropyLoss()
    total, correct, loss_sum = 0, 0, 0.0
    for img, lm, y in loader:
        img, lm, y = img.to(device), lm.to(device), y.to(device)
        optim.zero_grad()
        logits = model(img, lm)
        loss = ce(logits, y)
        loss.backward()
        optim.step()

        loss_sum += loss.item() * y.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    return loss_sum / max(total, 1), correct / max(total, 1)


In [4]:
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

num_classes = 7
batch_size = 64
lr = 1e-3
epochs = 40

In [5]:
def main():
    train_ds = FaceDataset(
        img_dir="face_images/train",
        landmarks_path="train_landmarks.npy",
        labels_path="train_labels.npy",
        ids_path="train_ids.npy",
    )
    val_ds = FaceDataset(
        img_dir="face_images/val",
        landmarks_path="val_landmarks.npy",
        labels_path="val_labels.npy",
        ids_path="val_ids.npy",
    )

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)

    model = CNN_LSTM_Fusion(num_classes=num_classes, lstm_hidden=128).to(device)
    optim = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optim, mode='max', patience=3, factor=0.5)

    print("Device:", device)
    best_acc = 0.0
    for epoch in tqdm(range(1, epochs + 1), desc="training"):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optim)
        va_loss, va_acc = evaluate(model, val_loader)
        scheduler.step(va_acc)

        print(f"Epoch {epoch:02d} | "
              f"train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
              f"val loss {va_loss:.4f} acc {va_acc:.4f} | "
              f"lr {optim.param_groups[0]['lr']:.6f}")
        
        if va_acc > best_acc:
            best_acc = va_acc
            torch.save(model.state_dict(), "best_model.pth")
            print(f"Saved best model (val acc = {best_acc:.4f})")

if __name__ == "__main__":
    main()

Device: cuda


training:   0%|          | 0/40 [00:00<?, ?it/s]

Epoch 01 | train loss 1.8160 acc 0.2546 | val loss 1.7982 acc 0.2625 | lr 0.001000
Saved best model (val acc = 0.2625)
Epoch 02 | train loss 1.8024 acc 0.2623 | val loss 1.7750 acc 0.2743 | lr 0.001000
Saved best model (val acc = 0.2743)
Epoch 03 | train loss 1.7746 acc 0.2743 | val loss 1.7409 acc 0.2935 | lr 0.001000
Saved best model (val acc = 0.2935)
Epoch 04 | train loss 1.6977 acc 0.3096 | val loss 1.6052 acc 0.3556 | lr 0.001000
Saved best model (val acc = 0.3556)
Epoch 05 | train loss 1.5891 acc 0.3622 | val loss 1.5117 acc 0.3947 | lr 0.001000
Saved best model (val acc = 0.3947)
Epoch 06 | train loss 1.5201 acc 0.3967 | val loss 1.4745 acc 0.3991 | lr 0.001000
Saved best model (val acc = 0.3991)
Epoch 07 | train loss 1.4737 acc 0.4150 | val loss 1.4172 acc 0.4424 | lr 0.001000
Saved best model (val acc = 0.4424)
Epoch 08 | train loss 1.4324 acc 0.4324 | val loss 1.3958 acc 0.4459 | lr 0.001000
Saved best model (val acc = 0.4459)
Epoch 09 | train loss 1.4001 acc 0.4493 | val lo